# 3. Improvement: or-opt, 2-opt, random restarts
**Class:** Session 3 slides 31–41 (2-opt, 3-opt, or-opt with p = 3, 2, 1), Session 10 slides 36–38 (neighbourhood search, neighbourhood size, random restart).

**IIMA context:** or-opt is "move this block of 1–3 topics earlier or later". It is the natural edit a student makes to a plan, limited to the window between the block's last prerequisite and first dependent.

In [1]:
import os, sys
ROOT = os.path.abspath('..'); sys.path.insert(0, os.path.join(ROOT, 'src'))
from speedrun.data import load_web, Params, Profile
from speedrun.graph import build_instance
from speedrun.costs import Evaluator
web = load_web(os.path.join(ROOT, 'data', 'sample'))
P = Params.load(os.path.join(ROOT, 'data', 'sample', 'params.json'))
inst = build_instance(web, Profile(), P); ev = Evaluator(inst)
name = lambda ids: ' -> '.join(ids)
print(ev.n, 'topics,', round(ev.sum_p, 1), 'study hours (estimates)')

28 topics, 71.5 study hours (estimates)


In [2]:
from speedrun import heuristics as H
s = H.nearest_neighbour(ev); c = ev.cost(s); print('NN start       ', round(c, 3))
s, c, _ = H.or_opt(ev, s, c); print('after or-opt   ', round(c, 3))
s, c, _ = H.two_opt(ev, s, c); print('after 2-opt    ', round(c, 3))
s, c = H.local_search(ev, s); print('local optimum  ', round(c, 3))
log = []; best_s, best = H.solve_heuristic(ev, restarts=8, log=log)
print('best of', len(log), 'starts:', round(best, 3))
for r in log: print(f"  {r['start']:<6} {r['construct']:7.2f} -> {r['improved']:7.2f}")

NN start        72.772
after or-opt    68.19
after 2-opt     68.19
local optimum   67.845
best of 12 starts: 67.83
  NN       72.77 ->   67.84
  CI       74.43 ->   69.56
  FI       74.03 ->   68.08
  SAV      72.05 ->   68.14
  rand0    90.52 ->   67.91
  rand1    88.75 ->   68.08
  rand2    85.64 ->   67.91
  rand3    88.11 ->   68.08
  rand4    80.99 ->   67.91
  rand5    85.62 ->   67.83
  rand6    87.87 ->   68.14
  rand7    88.91 ->   68.58


In an asymmetric, precedence-constrained instance 2-opt rarely applies. Reversing a segment breaks any prerequisite chain inside it, so or-opt does most of the work.